# 08 — Reshaping

Cambiar la forma del DataFrame: de ancho a largo, de largo a ancho, transponer, explotar listas.

## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT   = find_project_root()
TRAIN  = ROOT / 'data' / 'external' / 'train.csv'
AIR    = ROOT / 'data' / 'external' / 'Listings.csv'
df = pd.read_csv(TRAIN, low_memory=False)
air = pd.read_csv(AIR, encoding='latin-1', low_memory=False)


## melt() — de ancho a largo

Convierte columnas en filas. Útil cuando los datos tienen una columna por período (ene, feb, mar…) y se necesita una columna `mes` y una columna `valor`.

In [2]:
# Crear un DataFrame ancho de ejemplo — ventas por región y trimestre
ventas_wide = pd.DataFrame({
    'region': ['West', 'East', 'South', 'Central'],
    'Q1':     [240000, 185000, 98000, 130000],
    'Q2':     [270000, 210000, 112000, 145000],
    'Q3':     [225000, 195000, 105000, 138000],
    'Q4':     [310000, 230000, 120000, 160000],
})
print('Formato ancho:')
print(ventas_wide)

# melt: id_vars = columnas que se mantienen, value_vars = columnas que se convierten en filas
ventas_long = ventas_wide.melt(
    id_vars='region',
    value_vars=['Q1', 'Q2', 'Q3', 'Q4'],
    var_name='trimestre',
    value_name='ventas'
)
print()
print('Formato largo:')
print(ventas_long.sort_values(['region', 'trimestre']))


Formato ancho:
    region      Q1      Q2      Q3      Q4
0     West  240000  270000  225000  310000
1     East  185000  210000  195000  230000
2    South   98000  112000  105000  120000
3  Central  130000  145000  138000  160000

Formato largo:
     region trimestre  ventas
3   Central        Q1  130000
7   Central        Q2  145000
11  Central        Q3  138000
15  Central        Q4  160000
1      East        Q1  185000
5      East        Q2  210000
9      East        Q3  195000
13     East        Q4  230000
2     South        Q1   98000
6     South        Q2  112000
10    South        Q3  105000
14    South        Q4  120000
0      West        Q1  240000
4      West        Q2  270000
8      West        Q3  225000
12     West        Q4  310000


## pivot() — de largo a ancho

El opuesto de melt. Requiere que la combinación de index + columns sea única.

In [3]:
# Invertir el melt anterior
ventas_recuperada = ventas_long.pivot(
    index='region',
    columns='trimestre',
    values='ventas'
).reset_index()

ventas_recuperada.columns.name = None   # quitar el nombre del eje de columnas
print(ventas_recuperada)


    region      Q1      Q2      Q3      Q4
0  Central  130000  145000  138000  160000
1     East  185000  210000  195000  230000
2    South   98000  112000  105000  120000
3     West  240000  270000  225000  310000


## pivot_table() — pivot con agregación

Cuando hay duplicados en la combinación index+columns, `pivot()` falla. `pivot_table()` los resuelve con una función de agregación.

In [4]:
# Ventas totales por Region y Category
tabla = pd.pivot_table(
    df,
    values='Sales',
    index='Region',
    columns='Category',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
).round(0)
print(tabla)
print()

# Múltiples valores de agregación
tabla2 = pd.pivot_table(
    df,
    values='Sales',
    index='Region',
    columns='Category',
    aggfunc=['sum', 'count']
)
print(tabla2.head(2))


Category  Furniture  Office Supplies  Technology      Total
Region                                                     
Central    160317.0         163590.0    168739.0   492647.0
East       206461.0         199941.0    263117.0   669519.0
South      116531.0         124425.0    148195.0   389151.0
West       245348.0         217467.0    247405.0   710220.0
Total      728659.0         705422.0    827456.0  2261537.0

                  sum                                 count                  \
Category    Furniture Office Supplies  Technology Furniture Office Supplies   
Region                                                                        
Central   160317.4622      163590.243  168739.208       470            1399   
East      206461.3880      199940.811  263116.527       591            1667   

                     
Category Technology  
Region               
Central         408  
East            527  


## crosstab() — tabla de frecuencias

In [5]:
# Frecuencia de combinaciones entre dos columnas categóricas
tabla = pd.crosstab(
    df['Region'],
    df['Category'],
    margins=True
)
print(tabla)
print()

# normalize='index' para proporciones por fila
tabla_pct = pd.crosstab(
    df['Region'],
    df['Category'],
    normalize='index'
).round(3)
print(tabla_pct)


Category  Furniture  Office Supplies  Technology   All
Region                                                
Central         470             1399         408  2277
East            591             1667         527  2785
South           326              983         289  1598
West            691             1860         589  3140
All            2078             5909        1813  9800

Category  Furniture  Office Supplies  Technology
Region                                          
Central       0.206            0.614       0.179
East          0.212            0.599       0.189
South         0.204            0.615       0.181
West          0.220            0.592       0.188


## explode() — expandir listas en filas

Cuando una celda contiene una lista, `explode()` crea una fila por elemento de la lista.

In [6]:
# Simular una columna con listas (como amenities en Airbnb)
df_amenities = air[['listing_id', 'amenities']].head(5).copy()
df_amenities['amenities'] = df_amenities['amenities'].str.split(',')
print('Antes:')
print(df_amenities)

# explode — una fila por amenity
df_exploded = df_amenities.explode('amenities')
df_exploded['amenities'] = df_exploded['amenities'].str.strip().str.strip('"[]')
print()
print('Después:')
print(df_exploded.head(15))


Antes:


   listing_id                                          amenities
0      281420  [["Heating",  "Kitchen",  "Washer",  "Wifi",  ...
1     3705183  [["Shampoo",  "Heating",  "Kitchen",  "Essenti...
2     4082273  [["Heating",  "TV",  "Kitchen",  "Washer",  "W...
3     4797344  [["Heating",  "TV",  "Kitchen",  "Wifi",  "Lon...
4     4823489  [["Heating",  "TV",  "Kitchen",  "Essentials",...

Después:
   listing_id                amenities
0      281420                  Heating
0      281420                  Kitchen
0      281420                   Washer
0      281420                     Wifi
0      281420  Long term stays allowed
1     3705183                  Shampoo
1     3705183                  Heating
1     3705183                  Kitchen
1     3705183               Essentials
1     3705183                   Washer
1     3705183                    Dryer
1     3705183                     Wifi
1     3705183  Long term stays allowed
2     4082273                  Heating
2     4082273 

---
## Resumen

| Operación | Sintaxis | Caso de uso |
|-----------|----------|-------------|
| Columnas → filas | `df.melt(id_vars, value_vars)` | Normalizar datos anchos |
| Filas → columnas | `df.pivot(index, columns, values)` | Sin duplicados |
| Filas → columnas (con agg) | `pd.pivot_table(df, values, index, columns, aggfunc)` | Con duplicados |
| Frecuencias cruzadas | `pd.crosstab(col1, col2)` | Análisis categórico |
| Lista → filas | `df.explode('col')` | Columnas con listas |
